In [1]:
import os
import sys

current_path = os.path.abspath(os.getcwd())
base_path = os.path.abspath(os.path.join(current_path, '../../../'))
sys.path.append(base_path)

import tensorflow as tf
from tensorflow.keras.callbacks import TensorBoard

import numpy as np
from tqdm import tqdm
import datetime

from data_loaders.data_loader_patches import DataLoaderPatches
import project_images_loader as pil
import matplotlib.pyplot as plt

import discriminator as Discriminator
import generator as Generator
from layers.Wavelet3D import DWT3D

import utils.functions as fc
import utils.util_deep_learning as udl
import utils.visualization as vis
import utils.util as util
import utils.patches_controller as patches_controller



2024-03-30 02:31:15.653814: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-03-30 02:31:15.693848: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-03-30 02:31:15.693877: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-03-30 02:31:15.695093: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-03-30 02:31:15.702461: I tensorflow/core/platform/cpu_feature_guar

## Visualize data

In [2]:

# %matplotlib widget

# data_train = DataLoader(batch_size=3, image_size=(193,229,193), shuffle=False, channels_first=False, image_sufix=pil.INAME_PREPROCESSED, nb_patients=-1, server=False)
# fc.imgshow_3D("", data_train[0][0][0,0])



### Validation visualization

In [3]:

def generate_image_step(img_org, G, patch_size, patch_step, batch_size=20):
    img_patches = patches_controller.extract_patches(img_org, patch_shape=patch_size, extraction_step=patch_step).astype(np.float32)
    gen_patches = np.zeros_like(img_patches)

    for i in tqdm(range(0, len(img_patches), batch_size)):
        patches_batch = np.expand_dims(img_patches[i:min(i+batch_size, len(img_patches))], axis=-1)
        gen_result = G(patches_batch, training=False)
        gen_patches[i:min(i+batch_size, len(img_patches))] = gen_result[-1][..., 0]

    gen_image = patches_controller.perform_voting(gen_patches, 
                                                  patch_shape=patch_size, 
                                                  expected_shape=img_org.shape, 
                                                  extraction_step=patch_step,
                                                  window_sep=patch_step)
    return gen_image


def show_generated_image(id, G_h4, G_h3, patch_size, patch_step):
    img_hunt3, _ = pil.load_preprocessed_image(3, id, absolute=True)
    img_hunt4, _ = pil.load_preprocessed_image(4, id, absolute=True)

    img_gen_h4 = generate_image_step(img_hunt3, G_h4, patch_size=patch_size, patch_step=patch_step)
    img_gen_h3 = generate_image_step(img_hunt4, G_h3, patch_size=patch_size, patch_step=patch_step)
   
    vis.shows_3_images(img_hunt3, img_hunt4, img_gen_h4)
    vis.shows_3_images(img_hunt3, img_hunt4, img_gen_h3)


### Loss functions

In [4]:
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

# def adversarial_loss(disc_h4_real, disc_h3_real, disc_h4_fake, disc_h3_fake):
#     loss = (
#         tf.reduce_mean(tf.math.log(disc_h3_real)) +
#         tf.reduce_mean(tf.math.log(1.0 - disc_h3_fake)) +
#         tf.reduce_mean(tf.math.log(disc_h4_real)) +
#         tf.reduce_mean(tf.math.log(1.0 - disc_h4_fake))
#     )
#     return loss

def adversarial_loss(disc_h4_real, disc_h3_real, disc_h4_fake, disc_h3_fake):

    real_loss_h3 = cross_entropy(tf.ones_like(disc_h3_real), disc_h3_real)
    real_loss_h4 = cross_entropy(tf.ones_like(disc_h4_real), disc_h4_real)

    fake_loss_h3 = cross_entropy(tf.zeros_like(disc_h3_fake), disc_h3_fake)
    fake_loss_h4 = cross_entropy(tf.zeros_like(disc_h4_fake), disc_h4_fake)

    loss = real_loss_h3 + real_loss_h4 + fake_loss_h3 + fake_loss_h4
    return loss

def quality_loss(target_image, predicted_image, disc_prob, beta=1.5):
    loss = tf.reduce_mean(tf.abs(target_image - predicted_image) * tf.pow(1 - disc_prob, beta))
    return loss

def gram_matrix(input_tensor):
    batch_size, width, height, deep, channels = input_tensor.shape
    reshaped_tensor = tf.reshape(input_tensor, [batch_size, width * height, deep])
    gram_matrix = tf.matmul(reshaped_tensor, reshaped_tensor, transpose_a=True)
    num_elements = tf.cast(width * height * deep * channels, tf.float32)
    gram_matrix /= num_elements
    return gram_matrix


def texture_loss(target_image, predicted_image):
    gram_real = gram_matrix(target_image)
    gram_fake = gram_matrix(predicted_image)
    
    loss = tf.reduce_mean(tf.square(gram_real - gram_fake))
    
    return loss

dwt3D_layer = DWT3D(wavelet_name="bior1.3")
def frequency_loss(target_image, predicted_image):
    dwt_target_image = dwt3D_layer(target_image)
    dwt_predicted_image = dwt3D_layer(predicted_image)
    loss = tf.reduce_mean(tf.abs(dwt_target_image - dwt_predicted_image))
    return loss

def calc_cycle_loss(target_image, cycled_image):
    loss1 = tf.reduce_mean(tf.abs(target_image - cycled_image))
    return loss1

2024-03-30 02:31:19.379820: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 865 MB memory:  -> device: 0, name: NVIDIA A100 80GB PCIe, pci bus id: 0000:c0:00.0, compute capability: 8.0


In [5]:
# @tf.function
def train_step(G_h3, D_h3, G_h4, D_h4, h3_batch, h4_batch):
    print("train_step")
    h4_fake1, h4_fake2, h4_fake3  = G_h4(h3_batch, training=True)
    h3_fake1_cyc, h3_fake2_cyc, h3_fake3_cyc = G_h3(h4_fake3, training=True)
    print("train_step 2")
    h3_fake1, h3_fake2, h3_fake3  = G_h3(h4_batch, training=True)
    h4_fake1_cyc, h4_fake2_cyc, h4_fake3_cyc = G_h4(h3_fake3, training=True)
    print("train_step 3")

    disc_h4_real1, disc_h4_real2, disc_h4_real3 = D_h4(h4_batch, training=True)
    disc_h4_fake1, disc_h4_fake2, disc_h4_fake3 = D_h4(h4_fake3, training=True)
    print("train_step 4")
    disc_h3_real1, disc_h3_real2, disc_h3_real3 = D_h3(h3_batch, training=True)
    disc_h3_fake1, disc_h3_fake2, disc_h3_fake3 = D_h3(h3_fake3, training=True)
    print("train_step 5")

    # obtain reduced batches
    h4_batch_2 = tf.nn.pool(h4_batch, window_shape=(2, 2, 2), pooling_type='AVG', strides=(2, 2, 2), padding='VALID')
    h4_batch_1 = tf.nn.pool(h4_batch_2, window_shape=(2, 2, 2), pooling_type='AVG', strides=(2, 2, 2), padding='VALID')
    
    h3_batch_2 = tf.nn.pool(h3_batch, window_shape=(2, 2, 2), pooling_type='AVG', strides=(2, 2, 2), padding='VALID')
    h3_batch_1 = tf.nn.pool(h3_batch_2, window_shape=(2, 2, 2), pooling_type='AVG', strides=(2, 2, 2), padding='VALID')


    # ---- Adversarial loss
    ls1_adv = adversarial_loss(disc_h4_real1, disc_h3_real1, disc_h4_fake1, disc_h3_fake1)
    ls2_adv = adversarial_loss(disc_h4_real2, disc_h3_real2, disc_h4_fake2, disc_h3_fake2)
    ls3_adv = adversarial_loss(disc_h4_real3, disc_h3_real3, disc_h4_fake3, disc_h3_fake3)

    print(f"Adv losses: {ls1_adv}, {ls2_adv}, {ls3_adv}")
    # ---- Paired loss
    # Quality loss
    ls1_qa_g_h4, ls1_qa_g_h3 = quality_loss(h4_batch_1, h4_fake1, disc_h4_fake1), quality_loss(h3_batch_1, h3_fake1, disc_h3_fake1)
    ls2_qa_g_h4, ls2_qa_g_h3 = quality_loss(h4_batch_2, h4_fake2, disc_h4_fake2), quality_loss(h3_batch_2, h3_fake2, disc_h3_fake2)
    ls3_qa_g_h4, ls3_qa_g_h3 = quality_loss(h4_batch, h4_fake3, disc_h4_fake3), quality_loss(h3_batch, h3_fake3, disc_h3_fake3)
    print(f"Quality losses H4: {ls1_qa_g_h4}, {ls2_qa_g_h4}, {ls3_qa_g_h4}")
    print(f"Quality losses H3: {ls1_qa_g_h3}, {ls2_qa_g_h3}, {ls3_qa_g_h3}")
    # Texture loss
    ls1_tex_g_h4, ls1_tex_g_h3 = texture_loss(h4_batch_1, h4_fake1), texture_loss(h3_batch_1, h3_fake1)
    ls2_tex_g_h4, ls2_tex_g_h3 = texture_loss(h4_batch_2, h4_fake2), texture_loss(h3_batch_2, h3_fake2)
    ls3_tex_g_h4, ls3_tex_g_h3 = texture_loss(h4_batch, h4_fake3), texture_loss(h3_batch, h3_fake3)
    print(f"Texture losses H4: {ls1_tex_g_h4}, {ls2_tex_g_h4}, {ls3_tex_g_h4}")
    print(f"Texture losses H3: {ls1_tex_g_h3}, {ls2_tex_g_h3}, {ls3_tex_g_h3}")
    # Frequency loss
    ls1_freq_g_h4, ls1_freq_g_h3 = frequency_loss(h4_batch_1, h4_fake1), frequency_loss(h3_batch_1, h3_fake1)
    ls2_freq_g_h4, ls2_freq_g_h3 = frequency_loss(h4_batch_2, h4_fake2), frequency_loss(h3_batch_2, h3_fake2)
    ls3_freq_g_h4, ls3_freq_g_h3 = frequency_loss(h4_batch, h4_fake3), frequency_loss(h3_batch, h3_fake3)
    print(f"Frequency losses H4: {ls1_freq_g_h4}, {ls2_freq_g_h4}, {ls3_freq_g_h4}")
    print(f"Frequency losses H3: {ls1_freq_g_h3}, {ls2_freq_g_h3}, {ls3_freq_g_h3}")
    # ---- Cycle consistency loss
    lsc1 = calc_cycle_loss(h4_batch_1, h4_fake1_cyc) + calc_cycle_loss(h3_batch_1, h3_fake1_cyc)
    lsc2 = calc_cycle_loss(h4_batch_2, h4_fake2_cyc) + calc_cycle_loss(h3_batch_2, h3_fake2_cyc)
    lsc3 = calc_cycle_loss(h4_batch, h4_fake3_cyc) + calc_cycle_loss(h3_batch, h3_fake3_cyc)
    print(f"Cycle consistency losses: {lsc1}, {lsc2}, {lsc3}")
    # ---- Total loss
    ls1_g_h4 = ls1_adv + ls1_qa_g_h4 + ls1_tex_g_h4 + ls1_freq_g_h4 + lsc1
    ls2_g_h4 = ls2_adv + ls2_qa_g_h4 + ls2_tex_g_h4 + ls2_freq_g_h4 + lsc2
    ls3_g_h4 = ls3_adv + ls3_qa_g_h4 + ls3_tex_g_h4 + ls3_freq_g_h4 + lsc3

    ls1_g_h3 = ls1_adv + ls1_qa_g_h3 + ls1_tex_g_h3 + ls1_freq_g_h3 + lsc1
    ls2_g_h3 = ls2_adv + ls2_qa_g_h3 + ls2_tex_g_h3 + ls2_freq_g_h3 + lsc2
    ls3_g_h3 = ls3_adv + ls3_qa_g_h3 + ls3_tex_g_h3 + ls3_freq_g_h3 + lsc3

    # ---- generator losses
    g_h4_loss = ls1_g_h4 + ls2_g_h4 + ls3_g_h4
    g_h3_loss = ls1_g_h3 + ls2_g_h3 + ls3_g_h3

    # ---- discriminator losses
    d_loss = ls1_adv + ls2_adv + ls3_adv

    return d_loss, g_h4_loss, g_h3_loss



### Training

In [6]:

def train(num_epochs, batch_size):
    image_size = (193,229,193)
    patch_size= (64,64,64)
    patch_step = (32,32,32)

    # ---- Models
    D_h4 = Discriminator.MetamorphicDiscriminator()
    G_h4 = Generator.MetamorphicGenerator()

    D_h3 = Discriminator.MetamorphicDiscriminator()
    G_h3 = Generator.MetamorphicGenerator()

    # ---- Data
    server = True
    data_train = DataLoaderPatches(pil.INAME_PREPROCESSED,
                                   batch_size=batch_size, 
                                   image_size=image_size, 
                                   patch_size=patch_size,
                                   patch_step=patch_step,
                                   images_onmemory=5,
                                   shuffle_patches=False,
                                   channels_first=False,
                                   background_th=0.15,
                                   shuffle_images=True,
                                   nb_patients=300,
                                    shuffle_before_selecting=False,
                                    server=server)


    for epoch in range(0,num_epochs):

        print(f"\nEpoch {epoch+1}/{num_epochs}")
        
        # visualize images
        # show_generated_image("08202", G_h4, G_h3, patch_size, patch_step)
        
        # ---- Train
        for step, (h3_batch, h4_batch) in enumerate(data_train):
            d_loss, g_h4_loss, g_h3_loss = train_step(G_h3, D_h3, G_h4, D_h4, h3_batch, h4_batch)
            print(f"Step {step}: D_loss={d_loss}, G_h4_loss={g_h4_loss}, G_h3_loss={g_h3_loss}")

train(num_epochs=50, batch_size=1)


Epoch 1/50
train_step


2024-03-30 02:31:30.169918: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
2024-03-30 02:31:30.242524: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2024-03-30 02:31:30.446267: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


train_step 2
train_step 3
train_step 4
train_step 5


"`binary_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Sigmoid activation and thus does not represent logits. Was this intended?


Adv losses: 3.216358184814453, 3.177661895751953, 3.031557321548462
Quality losses H4: 0.1789608597755432, 0.24395284056663513, 0.1458350419998169
Quality losses H3: 0.21617235243320465, 0.13200119137763977, 0.15727676451206207
Texture losses H4: 0.0006131164846010506, 0.00014874889166094363, 2.8043234578944976e-06
Texture losses H3: 0.0001580363023094833, 7.042443030513823e-05, 8.781616998021491e-06
Frequency losses H4: 0.3586640954017639, 0.2685931921005249, 0.21922719478607178
Frequency losses H3: 0.296650230884552, 0.27337849140167236, 0.23874539136886597
Cycle consistency losses: 0.9554264545440674, 1.0016157627105713, 0.7280164361000061
Step 0: D_loss=9.425577163696289, G_h4_loss=13.526634216308594, G_h3_loss=13.425098419189453
train_step
train_step 2
train_step 3
train_step 4
train_step 5
Adv losses: 3.2181406021118164, 3.180539131164551, 3.037299633026123
Quality losses H4: 0.1844942420721054, 0.2347334623336792, 0.1455659568309784
Quality losses H3: 0.21667899191379547, 0.1269

In [ ]:
# Sim: 0.00000 	 gen_loss: 0.93415 	 disc_loss: 1.11709: 100%|██████████| 8/8 [00:14<00:00,  1.86s/it]
# Sim: 0.00000 	 gen_loss: 1.22538 	 disc_loss: 1.64933: 100%|██████████| 8/8 [00:14<00:00,  1.86s/it]